# n-gram language models — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Predict the next token from a short count-based history
The notebook turns corpus counts into unigram, bigram and smoothed probabilities, then shows perplexity and generated paths.

In [ ]:
corpus='a b a b a c'.split(); vocab=sorted(set(corpus)); counts=[corpus.count(w) for w in vocab]
plt.figure(figsize=(5,3)); plt.bar(vocab,counts); plt.title('Unigram counts')
assert counts[vocab.index('a')]==3

In [ ]:
bigrams=list(zip(corpus,corpus[1:])); nexts={w:{v:0 for v in vocab} for w in vocab}
for a,b in bigrams: nexts[a][b]+=1
row=np.array([nexts['a'][v] for v in vocab])
plt.figure(figsize=(5,3)); plt.bar(vocab,row/row.sum()); plt.title('P(next | a) from bigram counts')
assert np.allclose(row/row.sum(), [0,0.667,0.333], atol=0.001)

In [ ]:
alpha=1; smooth=(row+alpha)/(row.sum()+alpha*len(vocab))
plt.figure(figsize=(5,3)); plt.bar(vocab,smooth); plt.title('Add-one smoothing gives unseen events mass')
assert smooth[0]>0 and abs(smooth.sum()-1)<1e-9

In [ ]:
seq='a b a c'.split(); probs=[nexts[a][b]/sum(nexts[a].values()) for a,b in zip(seq,seq[1:])]
ppl=math.exp(-np.mean(np.log(probs)))
plt.figure(figsize=(5,3)); plt.bar(['p1','p2','p3'],probs); plt.title(f'Perplexity = {ppl:.2f}')
assert abs(ppl-1.6509636244473134)<1e-9

In [ ]:
state='a'; path=[state]
for _ in range(4):
    p=np.array([nexts[state][v] for v in vocab],float); p=(p+1)/(p.sum()+len(vocab)); state=vocab[int(np.argmax(p))]; path.append(state)
plt.figure(figsize=(6,2)); plt.plot(range(len(path)),[vocab.index(x) for x in path],'-o'); plt.yticks(range(len(vocab)),vocab); plt.title('Greedy count model generation')
assert path[:3]==['a','b','a']